In [1]:
import pandas as pd
df=pd.read_csv(r"C:\Users\HP\Documents\Solar_Power_Generation_Prediction\data\solar data.csv")

In [3]:
df.head()

,Date,Time,DateTime,Country,DC_Power,AC_Power,Ambient_Temperature,Module_Temperature,Irradiation,Day,Month,Hour,Daily_Yield,Total_Yield
0,2024-01-01,0,2024-01-01 00:00:00,Norway,0.0,0.0,-10.15,-3.33,0.039,1,1,0,145.722,145.722
1,2024-01-01,1,2024-01-01 01:00:00,Norway,0.0,0.0,-10.06,-3.50,0.014,1,1,1,145.722,291.444
2,2024-01-01,2,2024-01-01 02:00:00,Norway,0.0,0.0,-9.95,-5.17,0.025,1,1,2,145.722,437.166
3,2024-01-01,3,2024-01-01 03:00:00,Norway,0.0,0.0,-9.27,-4.53,0.000,1,1,3,145.722,582.888
4,2024-01-01,4,2024-01-01 04:00:00,Norway,0.0,0.0,-9.62,-6.34,0.000,1,1,4,145.722,728.610


In [4]:
df.dtypes

Date                    object
Time                     int64
DateTime                object
Country                 object
DC_Power               float64
AC_Power               float64
Ambient_Temperature    float64
Module_Temperature     float64
Irradiation            float64
Day                      int64
Month                    int64
Hour                     int64
Daily_Yield            float64
Total_Yield            float64
dtype: object

In [5]:
df["Date"]=pd.to_datetime(df["Date"])
df["DateTime"]=pd.to_datetime(df["DateTime"])

In [6]:
df.dtypes

Date                   datetime64[ns]
Time                            int64
DateTime               datetime64[ns]
Country                        object
DC_Power                      float64
AC_Power                      float64
Ambient_Temperature           float64
Module_Temperature            float64
Irradiation                   float64
Day                             int64
Month                           int64
Hour                            int64
Daily_Yield                   float64
Total_Yield                   float64
dtype: object

## Seperate X and y

In [16]:
X=df.drop(columns=["AC_Power","DC_Power","Date","Time","DateTime","Daily_Yield","Total_Yield"])
y=df["AC_Power"]

In [17]:
X.columns

Index(['Country', 'Ambient_Temperature', 'Module_Temperature', 'Irradiation',
       'Day', 'Month', 'Hour'],
      dtype='object')

## Train Test Split

In [19]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2)

In [22]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(28108, 7)
(7028, 7)
(28108,)
(7028,)


## Missing values

In [28]:
print("Missing values in training data\n",X_train.isnull().sum().sum())
print("Missing values in testing data\n",X_test.isnull().sum().sum())

Missing values in training data
 0
Missing values in testing data
 0


## Numerical and Categorical columns

In [30]:
numerical_col=X_train.select_dtypes(include=["int64","float64"]).columns
numerical_col

Index(['Ambient_Temperature', 'Module_Temperature', 'Irradiation', 'Day',
       'Month', 'Hour'],
      dtype='object')

In [31]:
categorical_col=X_train.select_dtypes(include=["object","category"]).columns
categorical_col

Index(['Country'], dtype='object')

## Categorical Encoding

In [33]:
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(handle_unknown="ignore",sparse_output=False)
encoder

,categories,'auto'
,drop,None
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'ignore'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


#### Applying encoding to training data

In [36]:
encoded_country=encoder.fit_transform(X_train[["Country"]])
encoded_country

array([[1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       ...,
       [0., 0., 1., 0.],
       [1., 0., 0., 0.],
       [0., 0., 1., 0.]], shape=(28108, 4))

#### Checking encoded column names

In [37]:
encoder.get_feature_names_out(["Country"])

array(['Country_Australia', 'Country_Brazil', 'Country_India',
       'Country_Norway'], dtype=object)

#### Encoding testing data

In [39]:
encoding_test=encoder.transform(X_test[["Country"]])
encoding_test

array([[0., 0., 0., 1.],
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       ...,
       [0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 1., 0., 0.]], shape=(7028, 4))

In [45]:
encoded_view=pd.DataFrame(encoded_country.astype(int),columns=encoder.get_feature_names_out(["Country"]))
encoded_view

,Country_Australia,Country_Brazil,Country_India,Country_Norway
0,1,0,0,0
1,1,0,0,0
2,0,0,0,1
3,0,1,0,0
4,0,0,1,0
...,...,...,...,...
28103,0,0,1,0
28104,0,1,0,0
28105,0,0,1,0
28106,1,0,0,0


## Feature Scaling

In [50]:
from sklearn.preprocessing import StandardScaler
numerical_features=X_train.select_dtypes(include=["int64","float64"]).columns
scaler=StandardScaler()

In [51]:
sca_train=scaler.fit_transform(X_train[numerical_features])
sca_train

array([[ 0.82168959,  0.73366931,  0.47163202, -0.76164533, -0.43815669,
        -0.36425873],
       [-0.57156218, -0.57608038,  0.48276608, -1.66870577,  1.30249746,
        -0.5085739 ],
       [-1.59626007, -1.54598494, -0.83940415, -0.87502789,  1.01238843,
        -0.79720425],
       ...,
       [ 1.23241212,  1.07267324, -0.7614657 ,  1.27924065, -0.72826572,
        -1.0858346 ],
       [-0.95192329, -0.77460378, -0.77259976, -0.76164533, -1.59859279,
         1.22320818],
       [ 1.55711063,  1.61945378, -0.83940415, -0.30811512, -0.14804767,
        -1.23014977]], shape=(28108, 6))

In [53]:
sca_test=scaler.transform(X_test[numerical_features])
sca_test

array([[-2.40252441, -2.32325451,  1.98308136,  1.05247554,  1.30249746,
         0.06868679],
       [ 1.48879743,  1.48317925,  0.50503421, -0.53488022,  0.14206136,
        -0.5085739 ],
       [ 0.37723216,  0.22390161, -0.81991954,  1.50600576, -1.30848377,
         1.6561537 ],
       ...,
       [ 0.25156962,  0.18688877, -0.83940415,  0.82571043,  1.01238843,
        -0.94151942],
       [ 0.09638903,  0.13221072, -0.0433185 ,  0.59894532,  1.30249746,
         0.79026266],
       [ 0.17060583,  0.17258836,  0.27400236, -1.101793  ,  1.30249746,
        -0.5085739 ]], shape=(7028, 6))

In [54]:
from sklearn.compose import ColumnTransformer
preprocess=ColumnTransformer(transformers=[("number",StandardScaler(), numerical_features),("category", OneHotEncoder(handle_unknown="ignore"), ["Country"])])
preprocess

,transformers,"[('number', ...), ('category', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True
